# Exercise 1 — parse_score and build_sentiment_prompt

These two functions are the foundation of the sentiment pipeline. `build_sentiment_prompt` wraps a headline in the message format the LLM expects. `parse_score` extracts a float from whatever the LLM returns — because LLMs don't always follow instructions perfectly.

In [ ]:
import re

# Gate-safe mock LLM — keyword-based, deterministic, no Ollama required
# Checks only the user message to avoid matching keywords in the system prompt.
def _mock_llm(messages):
    user_text = next(
        (m.get("content", "") for m in messages if m.get("role") == "user"), ""
    ).lower()
    if any(w in user_text for w in ["surge", "rally", "rise", "gain", "bull", "strong"]):
        return "0.75"
    if any(w in user_text for w in ["crash", "fall", "decline", "bear", "weak", "loss"]):
        return "-0.60"
    return "0.10"

BULLISH_HEADLINES = [
    "Tech stocks rally on strong earnings",
    "Markets surge as Fed signals rate pause",
    "S&P 500 gains 2% on positive jobs data",
    "Bull market continues with broad gains",
]
BEARISH_HEADLINES = [
    "Markets crash amid recession fears",
    "Stocks fall sharply on weak economic data",
    "S&P 500 declines on hawkish Fed remarks",
    "Bear market deepens as losses mount",
]
NEUTRAL_HEADLINES = [
    "Markets trade sideways in quiet session",
    "Mixed signals leave investors cautious",
    "Stocks finish flat as investors await data",
]

def parse_score(text):
    """Extract the first number from LLM output and clamp to [-1.0, 1.0].

    Steps:
      1. matches = re.findall(r"-?\\d+(?:\\.\\d+)?", text)
      2. if not matches: return 0.0
      3. return max(-1.0, min(1.0, float(matches[0])))

    Handles: "0.8", " -0.5 ", "Score: 0.75", "1.9" (clamped → 1.0), "no num" → 0.0
    """
    # TODO: implement the 3 steps
    return 0.0


def build_sentiment_prompt(headline):
    """Build the two-message list for LLM sentiment scoring.

    Returns:
        [
            {"role": "system", "content": <instruction to score from -1.0 to 1.0>},
            {"role": "user",   "content": f"Headline: {headline}"},
        ]
    """
    # TODO: return the two-message list
    return [{"role": "user", "content": headline}]


### Checks

In [ ]:
checks = 0

# 1 — parse_score: extracts a plain float
try:
    assert abs(parse_score("0.8")  - 0.8)  < 1e-9
    assert abs(parse_score("-0.5") - (-0.5)) < 1e-9
    assert abs(parse_score("0")    - 0.0)  < 1e-9
    checks += 1; print("✅ 1 parse_score extracts plain floats correctly")
except Exception as e:
    print("❌ 1:", e)

# 2 — parse_score: works with surrounding text
try:
    assert abs(parse_score("Score: 0.75")      - 0.75) < 1e-9
    assert abs(parse_score("The score is -0.3") - (-0.3)) < 1e-9
    checks += 1; print("✅ 2 parse_score works with surrounding text")
except Exception as e:
    print("❌ 2:", e)

# 3 — parse_score: clamps out-of-range values
try:
    assert abs(parse_score("-1.9") - (-1.0)) < 1e-9, f"expected -1.0, got {parse_score('-1.9')}"
    assert abs(parse_score("1.5")  -   1.0)  < 1e-9, f"expected  1.0, got {parse_score('1.5')}"
    checks += 1; print("✅ 3 parse_score clamps to [-1.0, 1.0]")
except Exception as e:
    print("❌ 3:", e)

# 4 — parse_score: returns 0.0 for no-number input
try:
    assert parse_score("bullish sentiment") == 0.0
    assert parse_score("") == 0.0
    checks += 1; print("✅ 4 parse_score returns 0.0 when no number found")
except Exception as e:
    print("❌ 4:", e)

# 5 — build_sentiment_prompt: structure check
try:
    h = "Markets surge on strong earnings"
    p = build_sentiment_prompt(h)
    assert isinstance(p, list) and len(p) == 2, f"expected list of 2, got {type(p)}/{len(p)}"
    roles = [m["role"] for m in p]
    assert "system" in roles, "missing system message"
    assert "user"   in roles, "missing user message"
    user_msg = next(m for m in p if m["role"] == "user")
    assert h in user_msg["content"], "headline not in user message"
    checks += 1; print("✅ 5 build_sentiment_prompt: 2 messages, system+user, headline in user")
except Exception as e:
    print("❌ 5:", e)

print(f"\n{checks}/5 checks passed!")
